<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Inferential Statistics Refresher

*Session 4 · Notebook 01 · Lecture · Coach version*

## Overview

Inferential statistics is the branch of statistics that uses data from a **sample** to draw conclusions about a larger **population**. It moves beyond simply describing the data in front of us (descriptive statistics) to making educated, quantified guesses about the world we cannot fully observe.

This notebook refreshes the core inferential toolkit (populations and samples, sampling error, standard error and confidence intervals) and then adds two techniques that any data team relies on: identifying the **probability distribution** a variable follows, and **transforming** skewed variables so that standard methods apply. The concepts are general purpose; a dedicated section shows how they map onto risk analysis specifically.

## Learning Objectives

By the end of this notebook you will be able to:

- Distinguish a population from a sample and explain why sampling error is unavoidable.
- Compute the standard error of the mean and explain how sample size affects precision.
- Build and correctly interpret a confidence interval for a mean.
- Recognise and fit Normal, Log-Normal and Poisson distributions with `scipy.stats`.
- Apply log, Box-Cox and Yeo-Johnson transformations to reduce skew, and check the effect.

## Prerequisites

- Session 1 (Python fundamentals) and Session 2 (pandas: `read_csv`, `describe`, column selection).
- Comfort with NumPy arrays and basic matplotlib/seaborn plots.

## Index

1. [Why this matters for risk analysis](#sec1)
2. [Populations vs Samples](#sec2)
3. [Sampling Error and Standard Error](#sec3)
4. [Confidence Intervals](#sec4)
5. [Probability Distributions](#sec5)
6. [Data Transformations](#sec6)
7. [Exercises](#exercises)
8. [Additional Exercises](#additional)
9. [Challenge](#challenge)
10. [Key Takeaways](#takeaways)
11. [Further Reading](#reading)

### Descriptive vs Inferential Statistics

| Type | Purpose | Example |
|------|---------|---------|
| **Descriptive** | Summarise and describe data you have | "The average salary in our dataset is 45,000" |
| **Inferential** | Make predictions about data you don't have | "Based on our sample, the average salary in the population is likely 45,000 +/- 3,000" |


### Tools for Inferential Statistics in Python

| Library | Purpose | Example import |
|---------|---------|---------------|
| **pandas** | Data manipulation and summary statistics | `import pandas as pd` |
| **numpy** | Numerical computation on arrays | `import numpy as np` |
| **matplotlib** | Basic plotting | `import matplotlib.pyplot as plt` |
| **seaborn** | Statistical visualisation | `import seaborn as sns` |
| **scipy.stats** | Distributions, hypothesis tests, intervals | `from scipy import stats` |
| **scikit-learn** | Preprocessing such as power transforms | `from sklearn.preprocessing import PowerTransformer` |


<a id="setup"></a>
# Section 0: Setup

The teaching **examples** below use small, simulated datasets so that each idea is self-contained and the ideal shapes are clear. The **exercises** at the end use a real, general-purpose dataset (`superstore.csv`) so you can practise on messier data. We also load `Salaries.csv` as a simple right-skewed amount variable for the sampling examples. Both files are read from the repo-root `datasets/` folder.

**Documentation:** [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) - the core library for the distributions, intervals and tests used throughout this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import PowerTransformer

sns.set_theme(style='whitegrid')
np.random.seed(42)  # reproducible results throughout

# Or read directly from the public S3 bucket (no local file needed):
# salaries = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_4/Salaries.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# salaries = pd.read_csv(session_datasets_http["Salaries"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# salaries = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_4/Salaries.csv", header=True, inferSchema=True).toPandas()
salaries = pd.read_csv('../datasets/Session_4/Salaries.csv')
amount = pd.to_numeric(salaries['TotalPay'], errors='coerce').dropna()
amount = amount[amount > 0]

# Or read directly from the public S3 bucket (no local file needed):
# store = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_4/superstore.csv', encoding='latin1')   # used in the exercises
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# store = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_4/superstore.csv", header=True, inferSchema=True).toPandas()   # used in the exercises
store = pd.read_csv('../datasets/Session_4/superstore.csv', encoding='latin1')   # used in the exercises
print('amount values:', len(amount), '| superstore rows:', len(store))

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

The techniques in this notebook are general statistical tools, but they are the daily bread of a risk team. Risk is fundamentally about reasoning under uncertainty from incomplete information, which is exactly what inferential statistics is for.

| Concept in this notebook | How a risk team uses it |
|---|---|
| **Population vs sample** | We almost never see every loan, claim, transaction or trade. We estimate portfolio behaviour from a sample. |
| **Sampling error and standard error** | A default rate or fraud rate measured on a sample is an estimate, not a fact. SE tells us how much it could move. |
| **Confidence intervals** | Reporting a KPI as a range ("default rate 4.1%, 95% CI 3.6% to 4.6%") is more honest and defensible to regulators and management than a single number. |
| **Probability distributions** | Loss severity is typically Log-Normal, event counts (defaults, fraud cases, claims) are typically Poisson, and returns are often modelled as Normal. Choosing the right distribution drives capital and provisioning models. |
| **Data transformations** | Exposures, loss amounts and balances are heavily skewed. Transforming them is often required before regression, scorecards or hypothesis tests behave well. |

Keep this mapping in mind: every generic example below has a direct risk counterpart.

<a id="sec2"></a>
# Section 2: Populations vs Samples

Understanding the difference between populations and samples is fundamental to inferential statistics.

### Population

**Definition:** the complete set of all items or individuals you are interested in studying.

- Usually too large, expensive or impossible to study completely.
- Its true values (parameters) are typically unknown, which is what we are trying to estimate.
- Denoted with Greek letters: mu for the mean, sigma for the standard deviation.

### Sample

**Definition:** a subset of the population that you actually observe and measure.

- A manageable size for practical study.
- Its values (statistics) can be calculated directly: mean, standard deviation, and so on.
- Denoted with Latin letters: x-bar for the mean, s for the standard deviation.
- Used to estimate the population parameters.

**Analogy: tasting soup**

| Concept | General scenario | Analogy |
|---|---|---|
| **Population** | All transactions processed in a year | The entire pot of soup |
| **Sample** | A random selection of 1,000 transactions | A single spoonful you taste |
| **Inference** | Concluding the average across all transactions is near the sampled rate | Concluding the whole pot is seasoned correctly from the spoonful |


### Syntax

```python
series.mean()              # mean of whatever you pass (population or sample)
series.sample(n=100)       # draw a random sample of size n
series.sample(frac=0.05)   # draw 5% of the rows
```

### Example

We treat the full `amount` column as the population (so we know the true mean), then see how close a single random sample of 100 gets.

In [ ]:
population_mean = amount.mean()
sample = amount.sample(n=100, random_state=1)

print(f'Population mean: {population_mean:,.0f}')
print(f'Sample mean:     {sample.mean():,.0f}')
print(f'Difference:      {sample.mean() - population_mean:,.0f}')

### Try it yourself

Draw a random sample of **500** values from `amount` (use `random_state=7`) and print its mean. Is it closer to the population mean than the sample of 100 was?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
big_sample = amount.sample(n=500, random_state=7)
print(f'Sample (n=500) mean: {big_sample.mean():,.0f}')
print(f'Population mean:     {population_mean:,.0f}')

<a id="sec3"></a>
# Section 3: Sampling Error and Standard Error

### Sampling Error

**Definition:** the difference between a sample statistic and the true population parameter.

```
Sampling Error = Sample Statistic - Population Parameter = x-bar - mu
```

**Example:** a company surveyed 500 customers and found 60% satisfied, so it claimed "60% of all customers are satisfied". A second survey of 500 found 65%. That variation is sampling error, and it made the company realise it should report ranges, not exact numbers.

**Key concept:** sampling error is unavoidable. It is not a mistake, it is natural variation.

### Standard Error

The **standard error (SE)** quantifies sampling error: it is the standard deviation of the sampling distribution of the mean.

```
SE = sigma / sqrt(n)      (in theory, using the population sd)
SE = s / sqrt(n)          (in practice, using the sample sd s)
```

**Interpretation:**

- SE tells us how much sample means vary from sample to sample.
- Smaller SE means more precise estimates.
- SE decreases as the sample size increases.

**Reducing sampling error:** to cut the SE in half you must **quadruple** the sample size (because of the square root). More data buys precision, but at the cost of time and effort.

| Sample size (n) | SE if s = 10 | Precision |
|---|---|---|
| 25 | 10 / sqrt(25) = 2.0 | Low |
| 100 | 10 / sqrt(100) = 1.0 | Medium |
| 400 | 10 / sqrt(400) = 0.5 | High |


### Syntax

```python
s = series.std(ddof=1)            # sample standard deviation
se = s / np.sqrt(len(series))     # standard error by hand
se = stats.sem(series)            # standard error in one call
```

### Example

We compute the SE for one sample, then show empirically that it shrinks as `n` grows.

In [ ]:
sample = amount.sample(n=100, random_state=1)
print(f'SE (manual): {sample.std(ddof=1) / np.sqrt(len(sample)):,.1f}')
print(f'SE (scipy):  {stats.sem(sample):,.1f}')

print('\nHow SE falls as the sample grows:')
for n in [50, 100, 500, 1000, 5000]:
    s = amount.sample(n=n, random_state=0)
    print(f'  n={n:>5}: SE = {stats.sem(s):,.1f}')

### Try it yourself

Draw samples of size 30, 120 and 480 from `amount` (use `random_state=n` for each) and print the SE of each. Confirm that quadrupling `n` roughly halves the SE.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
for n in [30, 120, 480]:
    s = amount.sample(n=n, random_state=n)
    print(f'n={n:>4}: SE = {stats.sem(s):,.1f}')

<a id="sec4"></a>
# Section 4: Confidence Intervals

**Definition:** a range of values that likely contains the true population parameter, with a specified level of confidence.

**Example:** instead of saying "average customer spend is 45", a company reports "we are 95% confident average customer spend is between 42 and 48". Being explicit about uncertainty helps decision makers.

**Analogy:** a confidence interval is like a fisherman saying "I am 95% sure the fish is somewhere in this 5-metre radius" rather than pointing at one exact spot.

#### Understanding confidence levels

A 95% confidence interval means: if we took 100 different samples and built 100 intervals, about 95 of them would contain the true parameter. It does **not** mean there is a 95% chance the true value lies in this one interval (a common misconception). The confidence is in the *method*, which captures the true value 95% of the time.

#### Calculating a confidence interval

```
CI = x-bar  +/-  (t* x SE)
```

Where:
- x-bar = sample mean
- t* = critical t-value for the chosen confidence level and degrees of freedom
- SE = s / sqrt(n) = standard error

### Assumptions for a confidence interval

1. **Random sampling.** Observations are randomly selected and independent.
2. **Normality (or large n).** For n >= 30 the Central Limit Theorem makes the interval robust even if the data is not normal. For small n, the data should be approximately normal (check a histogram, Q-Q plot or Shapiro-Wilk test).
3. **No extreme outliers**, which would distort the mean and standard deviation.

---
## Worked Example: Plant Height Study

A botanist wants to estimate the average height of a new tomato plant variety. She randomly selects 18 plants and measures their heights (cm):

`45, 52, 48, 50, 46, 53, 49, 51, 47, 54, 48, 50, 52, 49, 46, 51, 48, 50`

She wants a **95% confidence interval** for the true average plant height. We work it through step by step, the way you would by hand, then reproduce it in one line.

**Step 1: sample mean (x-bar)**

```
x-bar = sum(x) / n = 889 / 18 = 49.39 cm
```

**Step 2: sample standard deviation (s)**

```
s = sqrt( sum((x - x-bar)^2) / (n - 1) ) = sqrt(108.24 / 17) = sqrt(6.37) = 2.52 cm
```

**Step 3: degrees of freedom (df)**

```
df = n - 1 = 18 - 1 = 17
```

**Step 4: critical t-value (t*)** for 95% with df = 17 is **2.110** (from a t-table or `stats.t.ppf`).

**Step 5: standard error (SE)**

```
SE = s / sqrt(n) = 2.52 / sqrt(18) = 0.59 cm
```

**Step 6: margin of error**

```
Margin = t* x SE = 2.110 x 0.59 = 1.25 cm
```

**Step 7: confidence interval**

```
CI = 49.39 +/- 1.25 = [48.14, 50.64] cm
```

**Interpretation:** we are 95% confident the true average height of all plants of this variety falls between 48.14 cm and 50.64 cm.

### In Python: the manual way and the one-line way

In [ ]:
heights = [45, 52, 48, 50, 46, 53, 49, 51, 47, 54, 48, 50, 52, 49, 46, 51, 48, 50]

# Manual, mirroring the steps above
x_bar = np.mean(heights)
s = np.std(heights, ddof=1)          # ddof=1 -> sample standard deviation
n = len(heights)
df = n - 1
t_star = stats.t.ppf(0.975, df)      # 95% -> 0.975 in each tail
se = s / np.sqrt(n)
margin = t_star * se
print(f'x-bar={x_bar:.2f}, s={s:.2f}, df={df}, t*={t_star:.3f}, SE={se:.2f}')
print(f'95% CI (manual): [{x_bar - margin:.2f}, {x_bar + margin:.2f}]')

# One line with scipy
lo, hi = stats.t.interval(0.95, df=df, loc=x_bar, scale=stats.sem(heights))
print(f'95% CI (scipy):  [{lo:.2f}, {hi:.2f}]')

Higher confidence requires a wider interval. Use 95% as the default; use 99% when mistakes are costly, and 90% when you can afford to be less conservative.

In [ ]:
for conf in [0.90, 0.95, 0.99]:
    lo, hi = stats.t.interval(conf, df=df, loc=x_bar, scale=stats.sem(heights))
    print(f'{conf:.0%} CI: [{lo:.2f}, {hi:.2f}]  width={hi-lo:.2f}')

### Try it yourself

Draw a sample of 200 values from `amount` (`random_state=3`) and build a **99%** confidence interval for the mean using `stats.t.interval`.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
s = amount.sample(n=200, random_state=3)
lo, hi = stats.t.interval(0.99, df=len(s)-1, loc=s.mean(), scale=stats.sem(s))
print(f'99% CI for the mean: [{lo:,.0f}, {hi:,.0f}]')

<a id="sec5"></a>
# Section 5: Probability Distributions

A **probability distribution** describes how the values of a variable are spread out: which values are common, which are rare, and with what probability.

### Why evaluate the distribution of a variable?

Before you run a test, build a model or quote a probability, you should know the shape of your data. Evaluating the distribution lets you:

- **Assign probabilities to outcomes.** Once you know the distribution you can answer questions like "how likely is a value above this threshold?".
- **Choose the correct method.** Many techniques (t-tests, linear regression, control limits) assume a particular shape, usually Normal. Using them on the wrong shape gives misleading results.
- **Simulate and stress-test.** Fitting a distribution lets you generate realistic scenarios (for example Monte Carlo simulation of losses).
- **Spot anomalies and set thresholds.** Knowing what is typical tells you what is genuinely unusual.
- **Summarise compactly.** A whole column can be described by a couple of parameters.

`scipy.stats` gives every distribution the same interface, which is worth memorising:

| Method | Returns |
|---|---|
| `.pdf(x)` / `.pmf(k)` | density (continuous) / probability (discrete) at a value |
| `.cdf(x)` | probability of being at or below `x` |
| `.ppf(q)` | the value at quantile `q` (inverse of cdf) |
| `.rvs(size=n)` | draw `n` random values |
| `.fit(data)` | estimate the distribution's parameters from data (continuous) |


## 5.1 The Normal distribution

**Definition:** a symmetric, bell-shaped distribution defined by its mean (mu) and standard deviation (sigma).

**Example:** heights of adults, measurement errors, exam scores across a large cohort.

**Analogy:** the result of many small, independent nudges. A person's height comes from many genes and environmental factors each pushing up or down a little; added together they pile up into a bell shape. This is the Central Limit Theorem at work.

**Explanation:** the Normal is the most important distribution in statistics because sums and averages of many independent effects tend toward it. The **68-95-99.7 rule** says about 68% of values fall within 1 sd of the mean, 95% within 2 sd and 99.7% within 3 sd. We simulate a sample, fit a Normal, overlay the fitted density, and read a **Q-Q plot** (if the points hug the diagonal, the data is close to Normal).

In [ ]:
# Simulated example: a generic measurement, such as a test score
data = np.random.normal(loc=70, scale=10, size=500)
mu, sigma = stats.norm.fit(data)
print(f'Fitted Normal: mean={mu:.2f}, sd={sigma:.2f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(data, bins=30, density=True, alpha=0.6, color='steelblue')
x = np.linspace(data.min(), data.max(), 200)
ax[0].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label='fitted Normal')
ax[0].set_title('Histogram vs fitted Normal'); ax[0].legend()
stats.probplot(data, dist='norm', plot=ax[1])
ax[1].set_title('Q-Q plot (Normal)')
plt.tight_layout(); plt.show()

print(f'P(value < 60) = {stats.norm.cdf(60, mu, sigma):.3f}')

### Reading the probability

The final line uses `stats.norm.cdf(60, mu, sigma)` to get `P(value < 60)`: the probability that a value drawn from the fitted Normal is below 60. The **cdf** (cumulative distribution function) is the area under the curve to the left of a value, so it always runs from 0 to 1. Here 60 is one standard deviation below the mean of 70, so about 16% of values fall below it. For the opposite question ("above 60") use `1 - cdf`, and for a range ("between a and b") use `cdf(b) - cdf(a)`.

In [ ]:
# The DIRECTION of a probability question (< vs >) is not a property of the distribution.
# It is always the same tool, the cdf (cumulative distribution function), read two ways:
#
#     cdf(x)      = P(value <= x)   ->  the area to the LEFT of x
#     1 - cdf(x)  = P(value >  x)   ->  the area to the RIGHT of x (the upper tail)
#
# The two are complementary, so they always add up to 1. That is why some examples in this
# notebook use cdf ("below") and others use 1 - cdf ("above"): they just ask the question
# in whichever direction is interesting. We demonstrate on the fitted Normal from 5.1
# (mean `mu`, standard deviation `sigma`).

x = 60
p_below = stats.norm.cdf(x, mu, sigma)         # P(value < 60): area to the LEFT  -> use cdf
p_above = 1 - stats.norm.cdf(x, mu, sigma)     # P(value > 60): area to the RIGHT -> use 1 - cdf
print(f'P(value < {x}) = {p_below:.3f}   (cdf: left tail)')
print(f'P(value > {x}) = {p_above:.3f}   (1 - cdf: right tail)')
print(f'They sum to 1 : {p_below + p_above:.3f}')

# A probability BETWEEN two values is just the difference of two cdfs:
#     P(a < value < b) = cdf(b) - cdf(a)
a, b = 60, 80
p_between = stats.norm.cdf(b, mu, sigma) - stats.norm.cdf(a, mu, sigma)
print(f'P({a} < value < {b}) = {p_between:.3f}   (cdf(b) - cdf(a))')

# Same idea for the other distributions: lognorm.cdf / poisson.cdf give "<=", and 1 - cdf gives ">".
# The only extra tool is for DISCRETE data (Poisson), where an EXACT count uses the pmf:
#     poisson.pmf(k) = P(exactly k)   |   poisson.cdf(k) = P(<= k)   |   1 - poisson.cdf(k) = P(> k)

## 5.2 The Log-Normal distribution

**Definition:** a variable is Log-Normal if its **logarithm** is Normally distributed. It is positive only and right-skewed (a long tail of large values).

**Example:** claim sizes, file sizes, or the time to complete a task. Most values are modest, with a few very large ones.

**Analogy:** incomes in a town. Everyone earns something positive, most people cluster around a modest wage, and a handful earn many times the typical amount, stretching a long right tail. Positive quantities that multiply rather than add tend to look like this.

**Explanation:** because it is bounded below by zero and skewed right, the Log-Normal fits amounts, claim sizes and durations far better than a Normal. We simulate a sample of amounts, fit a Log-Normal (`floc=0` fixes the location at zero, the standard choice for positive data) and use the fit to answer a probability question.

In [ ]:
# Simulated example: a generic positive amount (for example an income or a claim size)
data = np.random.lognormal(mean=10.5, sigma=0.5, size=2000)
shape, loc, scale = stats.lognorm.fit(data, floc=0)
print(f'Fitted Log-Normal: shape(sigma)={shape:.3f}, scale(exp mu)={scale:,.0f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(data, bins=60, density=True, alpha=0.6, color='seagreen')
x = np.linspace(data.min(), np.quantile(data, 0.99), 300)
ax.plot(x, stats.lognorm.pdf(x, shape, loc, scale), 'r-', lw=2, label='fitted Log-Normal')
ax.set_title('Amounts vs fitted Log-Normal'); ax.legend()
plt.show()

print(f'P(amount > 80,000) = {1 - stats.lognorm.cdf(80000, shape, loc, scale):.3f}')

### Reading the probability

`1 - stats.lognorm.cdf(80000, ...)` gives `P(amount > 80000)`: the area under the **right tail**, the chance a value exceeds 80,000. We take `1 - cdf` because `cdf(80000)` is the probability of being *below* 80,000, and the two must sum to 1. With a median near 36,000, a value above 80,000 sits well into the tail, so the probability is small (about 6%). This is the shape of many risk questions: "how likely is a loss or claim above this level?"

In [ ]:
# Same tool as the Normal, just on the Log-Normal fitted above (shape, loc, scale):
#     cdf(x)      = P(amount <= x)   -> area to the LEFT
#     1 - cdf(x)  = P(amount >  x)   -> area to the RIGHT (the upper tail)
# For a positive, right-skewed amount the RIGHT tail ("a large value") is usually the
# interesting risk question, but both directions come from the same fitted distribution.
x = 80000
p_below = stats.lognorm.cdf(x, shape, loc, scale)        # P(amount < 80,000)
p_above = 1 - stats.lognorm.cdf(x, shape, loc, scale)    # P(amount > 80,000): the tail
print(f'P(amount < {x:,}) = {p_below:.3f}   (cdf: left)')
print(f'P(amount > {x:,}) = {p_above:.3f}   (1 - cdf: right tail)')
print(f'They sum to 1    : {p_below + p_above:.3f}')

# A range is the difference of two cdfs: P(a < amount < b) = cdf(b) - cdf(a)
a, b = 40000, 80000
p_between = stats.lognorm.cdf(b, shape, loc, scale) - stats.lognorm.cdf(a, shape, loc, scale)
print(f'P({a:,} < amount < {b:,}) = {p_between:.3f}   (cdf(b) - cdf(a))')

## 5.3 The Poisson distribution

**Definition:** a discrete distribution for the **number of events in a fixed interval** of time or space, given a constant average rate `lambda`.

**Example:** customer arrivals at a counter per hour, defects per batch, support tickets per day.

**Analogy:** raindrops falling on equal-size paving squares during one minute. Each square catches a whole number of drops, most catch a few, some catch none, a rare one catches many. The counts per square follow a Poisson.

**Explanation:** Poisson has a single parameter `lambda` (the average count), and its mean and variance are both equal to `lambda`. It assumes events occur independently at a constant rate. We use `.pmf` (probability mass function) because outcomes are whole numbers.

In [ ]:
# Simulated example: number of events per interval, average rate lambda = 3
lam = 3
data = np.random.poisson(lam=lam, size=2000)
k = np.arange(0, data.max() + 1)
observed = pd.Series(data).value_counts(normalize=True).sort_index()

plt.figure(figsize=(8, 4))
plt.bar(observed.index, observed.values, alpha=0.6, color='steelblue', label='observed')
plt.plot(k, stats.poisson.pmf(k, mu=lam), 'ro-', label=f'Poisson(lambda={lam})')
plt.title('Simulated event counts vs Poisson'); plt.xlabel('events per interval')
plt.ylabel('probability'); plt.legend(); plt.show()

print(f'P(0 events)   = {stats.poisson.pmf(0, lam):.3f}')
print(f'P(more than 5)= {1 - stats.poisson.cdf(5, lam):.3f}')

### Reading the probabilities

Because counts are discrete, we use the **pmf** (probability mass function), which gives the probability of an *exact* whole-number count (not a density). `stats.poisson.pmf(0, lam)` is `P(exactly 0 events)`, a completely quiet interval (about 5% when lambda is 3). `1 - stats.poisson.cdf(5, lam)` is `P(more than 5 events)`: one minus the probability of 5 or fewer, so the chance of an unusually busy interval (about 8%). In short, `pmf` answers "exactly k", and `cdf` answers "at most k" (so `1 - cdf` answers "more than k").

In [ ]:
# Poisson is DISCRETE (whole-number counts), so it adds one tool, the pmf, on top of the cdf:
#     pmf(k)      = P(exactly k)
#     cdf(k)      = P(<= k)        (k or fewer)
#     1 - cdf(k)  = P(>  k)        (more than k, the upper tail)
# Watch the "off by one" with discrete counts: "at least k" is 1 - cdf(k - 1), NOT 1 - cdf(k).
k = 5
print(f'P(exactly {k}) = {stats.poisson.pmf(k, lam):.3f}   (pmf)')
print(f'P(<= {k})      = {stats.poisson.cdf(k, lam):.3f}   (cdf)')
print(f'P(> {k})       = {1 - stats.poisson.cdf(k, lam):.3f}   (1 - cdf)')
print(f'P(>= {k})      = {1 - stats.poisson.cdf(k - 1, lam):.3f}   (1 - cdf(k-1): at least k)')

# A range of counts, e.g. 2 to 5 inclusive: cdf(5) - cdf(1)
lo_k, hi_k = 2, 5
p_range = stats.poisson.cdf(hi_k, lam) - stats.poisson.cdf(lo_k - 1, lam)
print(f'P({lo_k} <= count <= {hi_k}) = {p_range:.3f}   (cdf(hi) - cdf(lo-1))')

### Estimating Poisson from a real dataset

In practice you are not handed `lambda`; you estimate it from the observed counts. For a Poisson the maximum-likelihood estimate is simply the **sample mean**, so there is no `.fit` to call (scikit-style fitting exists only for the continuous distributions). Below we treat the simulated counts above as if they were an observed dataset: recover `lambda` from the mean, then answer the same probability questions.

**Assumption to check (briefly):** a Poisson has **mean approximately equal to variance**. If the variance is much larger than the mean (overdispersion, common for bursty real-world counts), Poisson understates the tail and a **Negative Binomial** is the better choice.

In [ ]:
# Treat the counts above as an observed dataset (pretend we no longer know lambda)
lam_hat = data.mean()            # the MLE of lambda for a Poisson is just the sample mean
print(f'Estimated lambda (sample mean): {lam_hat:.2f}')
print(f'Mean = {data.mean():.2f}, Variance = {data.var():.2f}  (close -> Poisson is reasonable)')

# Answer probability questions using the ESTIMATED lambda
print(f'P(0 events)    = {stats.poisson.pmf(0, lam_hat):.3f}')
print(f'P(more than 5) = {1 - stats.poisson.cdf(5, lam_hat):.3f}')

### Try it yourself

Fit a Normal to the superstore `Discount` column with `stats.norm.fit`, then print the probability that a discount is **above 0.30** using the fitted `stats.norm.cdf`.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
mu, sigma = stats.norm.fit(store['Discount'])
print(f'Fitted mean={mu:.3f}, sd={sigma:.3f}')
print(f'P(discount > 0.30) = {1 - stats.norm.cdf(0.30, mu, sigma):.3f}')

<a id="sec6"></a>
# Section 6: Data Transformations

**Definition:** a transformation is a mathematical function applied to every value of a variable to reshape its distribution (most often, to make a skewed variable more symmetric).

**Example:** turning a heavily right-skewed amount into a roughly bell-shaped variable before fitting a model that assumes normality.

**Analogy:** switching to a fairer measuring scale. Earthquake energy (Richter) and sound (decibels) are reported on log scales precisely because the raw values span an enormous range; the log scale makes them readable and comparable.

### Why do we transform data?

- **Meet the assumptions** of t-tests, linear regression and confidence intervals, which expect roughly normal data.
- **Stabilise variance**, so that spread does not grow with the size of the values.
- **Linearise relationships**, making patterns easier to model.
- **Reduce the leverage of extreme values**, so a few huge observations do not dominate.

We measure skew with `stats.skew` (0 means symmetric, positive means a right tail). The aim of a transform is to pull skewness toward 0 and straighten the Q-Q plot.

| Transform | Function | Notes |
|---|---|---|
| **Log** | `np.log` / `np.log1p` | simple and interpretable; positive data only |
| **Box-Cox** | `stats.boxcox` | finds the best power automatically; strictly positive data only |
| **Yeo-Johnson** | `PowerTransformer` | like Box-Cox but also handles zeros and negatives |


We create one simulated right-skewed positive variable to use across the next two examples.

In [ ]:
# A generic, right-skewed positive variable (for example an amount or a duration)
skewed = np.random.lognormal(mean=9, sigma=1.0, size=2000)
print(f'Skew of the raw variable: {stats.skew(skewed):.2f}')

## 6.1 The log transform

**Definition:** replace each value `x` with `log(x)` (use `log1p`, which is `log(1 + x)`, when the data contains zeros).

**Example:** the log of an amount, which turns a long right tail into a near-symmetric spread.

**Analogy:** a camera that brightens the shadows while leaving the highlights alone. The log compresses large values much more than small ones, revealing detail across a huge range.

**Explanation:** the log works only on positive data and is highly interpretable: a fixed step on the log scale means a fixed *percentage* change on the original scale. It is the first transform to reach for with amounts and counts.

In [ ]:
log_skewed = np.log(skewed)
print(f'Skew raw      : {stats.skew(skewed):.2f}')
print(f'Skew log      : {stats.skew(log_skewed):.2f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(skewed, bins=60, color='salmon'); ax[0].set_title('Raw (right-skewed)')
ax[1].hist(log_skewed, bins=40, color='seagreen'); ax[1].set_title('After log')
plt.tight_layout(); plt.show()

### Computing a probability after the log transform

The transform is monotonic, so `P(X < x) = P(log X < log x)`: we fit a Normal to the log values, transform the threshold with `np.log`, and read the probability from the Normal cdf. We compare it to the empirical share below the threshold as a check. (This is only trustworthy when the transformed data really is close to Normal, so check the skew or Q-Q first.)

In [ ]:
x = 20000
mu, sigma = stats.norm.fit(log_skewed)            # Normal fitted on the log scale
# Transform the threshold with the SAME function (log), then read the cdf both ways:
p_below = stats.norm.cdf(np.log(x), mu, sigma)    # P(value < x): area to the LEFT
p_above = 1 - stats.norm.cdf(np.log(x), mu, sigma)  # P(value > x): area to the RIGHT (upper tail)
print(f'P(value < {x:,}) model: {p_below:.3f}  | empirical: {(skewed < x).mean():.3f}')
print(f'P(value > {x:,}) model: {p_above:.3f}  | empirical: {(skewed > x).mean():.3f}')

## 6.2 The Box-Cox transform

**Definition:** a family of power transforms controlled by a parameter `lambda`, where `lambda` is chosen automatically to make the data as Normal as possible.

**Example:** `stats.boxcox(data)`, which returns the transformed values and the best lambda.

**Analogy:** an automatic dial that tries every strength of transform and stops at the setting that looks most bell-shaped.

**Explanation:** Box-Cox works on **strictly positive** data only. Special cases of lambda recover familiar transforms: lambda = 0 is the log, lambda = 0.5 is the square root, and lambda = 1 is no change. Because it optimises lambda from the data, it often beats a plain log.

In [ ]:
boxcox_skewed, lam = stats.boxcox(skewed)
print(f'Best lambda: {lam:.3f}')
print(f'Skew raw     : {stats.skew(skewed):.2f}')
print(f'Skew log     : {stats.skew(log_skewed):.2f}')
print(f'Skew Box-Cox : {stats.skew(boxcox_skewed):.2f}')

# Before/after histograms (as in 6.1), then a Q-Q plot to check normality
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(skewed, bins=60, color='salmon'); ax[0].set_title('Raw (right-skewed)')
ax[1].hist(boxcox_skewed, bins=40, color='seagreen'); ax[1].set_title('After Box-Cox')
stats.probplot(boxcox_skewed, dist='norm', plot=ax[2]); ax[2].set_title('Q-Q: Box-Cox (on the line = near-normal)')
plt.tight_layout(); plt.show()

### Computing a probability after the Box-Cox transform

The same recipe, but the threshold must be transformed with the **same lambda** that Box-Cox chose (`stats.boxcox([x], lmbda=lam)`), then read off the Normal cdf.

In [ ]:
x = 20000
mu, sigma = stats.norm.fit(boxcox_skewed)
x_bc = stats.boxcox([x], lmbda=lam)[0]            # transform the threshold with the fitted lambda
# Read the cdf both ways on the Box-Cox scale:
p_below = stats.norm.cdf(x_bc, mu, sigma)         # P(value < x): area to the LEFT
p_above = 1 - stats.norm.cdf(x_bc, mu, sigma)     # P(value > x): area to the RIGHT (upper tail)
print(f'P(value < {x:,}) model: {p_below:.3f}  | empirical: {(skewed < x).mean():.3f}')
print(f'P(value > {x:,}) model: {p_above:.3f}  | empirical: {(skewed > x).mean():.3f}')

## 6.3 The Yeo-Johnson transform

**Definition:** an extension of Box-Cox that works for any real values, including zeros and negatives.

**Example:** a profit-and-loss figure or a balance change, which can be negative, so Box-Cox cannot be applied.

**Analogy:** Box-Cox's more flexible sibling that does not refuse negative numbers.

**Explanation:** because real financial variables (profit, balance change, returns) often go negative, Yeo-Johnson is the safe default inside machine-learning pipelines. In scikit-learn it is available through `PowerTransformer`, which also centres and scales the result.

In [ ]:
# A generic skewed variable that includes negative values
net_change = np.random.lognormal(mean=3, sigma=1.0, size=2000) - 25
pt = PowerTransformer(method='yeo-johnson')
yj = pt.fit_transform(net_change.reshape(-1, 1)).ravel()
print(f'Has negatives? {(net_change < 0).any()}')
print(f'Skew raw         : {stats.skew(net_change):.2f}')
print(f'Skew Yeo-Johnson : {stats.skew(yj):.2f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(net_change, bins=60, color='salmon'); ax[0].set_title('Raw (has negatives)')
ax[1].hist(yj, bins=40, color='seagreen'); ax[1].set_title('After Yeo-Johnson')
plt.tight_layout(); plt.show()

### Computing a probability after the Yeo-Johnson transform

Yeo-Johnson handles zero and negative values, so we can ask, for example, the probability that `net_change` is below 0 (a loss). We transform the threshold with the fitted transformer (`pt.transform`), then use the Normal cdf. (`PowerTransformer` also standardizes, so the transformed values already have mean 0 and standard deviation 1.)

In [ ]:
x = 0
mu, sigma = stats.norm.fit(yj)
x_yj = pt.transform([[x]])[0, 0]                  # transform the threshold with the fitted transformer
# Read the cdf both ways on the Yeo-Johnson scale (x = 0 splits losses from gains):
p_below = stats.norm.cdf(x_yj, mu, sigma)         # P(value < 0): a loss
p_above = 1 - stats.norm.cdf(x_yj, mu, sigma)     # P(value > 0): a gain
print(f'P(value < {x}) model: {p_below:.3f}  | empirical: {(net_change < x).mean():.3f}')
print(f'P(value > {x}) model: {p_above:.3f}  | empirical: {(net_change > x).mean():.3f}')

## 6.4 A note on transforms and outliers

Analysing **outliers** (unusually large or small values) is good practice before modelling. A transform helps here: rules that assume a **symmetric** distribution, such as flagging anything "more than 3 standard deviations from the mean", misbehave on skewed data because the long tail inflates the standard deviation. Applying such a rule on a transformed (for example log) scale, then back-transforming the cut-off, keeps it valid.

We cover outlier detection properly in notebook 04_03 (z-score, IQR and Mahalanobis distance). The point to keep from this section is simply that skewed data should usually be transformed before any symmetric-distribution check.

### Try it yourself

Apply a log transform (`np.log`) to the `amount` salary values and print the skewness before and after. By how much did the skew fall?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
print(f'Before: {stats.skew(amount):.2f}')
print(f'After log: {stats.skew(np.log(amount)):.2f}')

<a id="exercises"></a>
# Section 7: Exercises

These exercises use the real `superstore` dataset loaded in Section 0.

### Exercise 1: Confidence interval from a sample

Draw a random sample of 150 rows from the superstore `Sales` column (use `store['Sales'].sample(n=150, random_state=10)`) and build a **95%** confidence interval for the mean sale using `stats.t.interval`.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
s = store['Sales'].sample(n=150, random_state=10)
lo, hi = stats.t.interval(0.95, df=len(s)-1, loc=s.mean(), scale=stats.sem(s))
print(f'95% CI for mean Sales: [{lo:,.2f}, {hi:,.2f}]')

### Exercise 2: Probability from a fitted distribution

Fit a Log-Normal to the superstore `Sales` column (`stats.lognorm.fit(store['Sales'], floc=0)`) and estimate the probability that a sale falls **between 100 and 300** using the cdf.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
shape, loc, scale = stats.lognorm.fit(store['Sales'], floc=0)
p = stats.lognorm.cdf(300, shape, loc, scale) - stats.lognorm.cdf(100, shape, loc, scale)
print(f'P(100 < Sales < 300) = {p:.3f}')

### Exercise 3: Compare two transforms

The superstore `Sales` column is positive and right-skewed. Apply both a log transform and a Box-Cox transform, and print the skewness of each. Which gets closest to 0?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
log_s = np.log(store['Sales'])
bc_s, lam = stats.boxcox(store['Sales'])
print(f'Skew raw     : {stats.skew(store["Sales"]):.2f}')
print(f'Skew log     : {stats.skew(log_s):.2f}')
print(f'Skew Box-Cox : {stats.skew(bc_s):.2f} (lambda={lam:.3f})')

<a id="additional"></a>
## Additional Exercises

### Exercise A1: Standard error and sample size

Draw samples of size 50, 200 and 800 from the superstore `Sales` column (use `random_state=n`) and print the SE of each. Confirm the SE roughly halves when `n` is multiplied by 4.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
for n in [50, 200, 800]:
    s = store['Sales'].sample(n=n, random_state=n)
    print(f'n={n:>4}: SE={stats.sem(s):,.2f}')

### Exercise A2: A transform for data with negatives

The superstore `Profit` column contains negative values, so Box-Cox cannot be used. Apply a Yeo-Johnson transform with `PowerTransformer` and print the skewness before and after.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
pt = PowerTransformer(method='yeo-johnson')
yj = pt.fit_transform(store['Profit'].values.reshape(-1, 1)).ravel()
print(f'Skew before: {stats.skew(store["Profit"]):.2f}')
print(f'Skew after Yeo-Johnson: {stats.skew(yj):.2f}')

<a id="challenge"></a>
## Challenge (optional): a mini distribution-and-transform study

Bring the whole notebook together on the superstore data. Work through the four parts; the coach answer shows one complete solution.

**Part 1: Identify and fit a distribution.** Plot a histogram of `Sales`, fit a Log-Normal with `stats.lognorm.fit(..., floc=0)`, overlay the fitted density, and use the fit to report the probability that a sale exceeds 1,000.

**Part 2: Transform a positive, skewed variable.** Apply both a log and a Box-Cox transform to `Sales`, compare the skewness of each, and state which you would use and why.

**Part 3: Transform a variable with negatives.** Explain in one line why Box-Cox cannot be applied to `Profit`, then apply a Yeo-Johnson transform and report the skewness before and after.

**Part 4: Compare confidence intervals on two scales.** From a sample of 300 sales, build a 95% CI for the mean of the raw `Sales` and a 95% CI for the mean of `log(Sales)`. Comment on which interval you would trust more for a small sample, and why.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
sales = store['Sales']

# Part 1: fit a Log-Normal and report a tail probability
shape, loc, scale = stats.lognorm.fit(sales, floc=0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sales, bins=80, density=True, alpha=0.6, color='seagreen')
x = np.linspace(sales.min(), sales.quantile(0.99), 300)
ax.plot(x, stats.lognorm.pdf(x, shape, loc, scale), 'r-', lw=2, label='fitted Log-Normal')
ax.set_title('Sales vs fitted Log-Normal'); ax.legend(); plt.show()
print(f'P(Sales > 1,000) = {1 - stats.lognorm.cdf(1000, shape, loc, scale):.3f}')

# Part 2: log vs Box-Cox on Sales
log_sales = np.log(sales)
bc_sales, lam = stats.boxcox(sales)
print(f'\nSkew raw={stats.skew(sales):.2f}, log={stats.skew(log_sales):.2f}, '
      f'Box-Cox={stats.skew(bc_sales):.2f} (lambda={lam:.3f})')
print('Box-Cox is closest to 0, so it is the better choice; log is simpler and almost as good.')

# Part 3: Profit has negatives, so Box-Cox (which needs x > 0) fails; use Yeo-Johnson
pt = PowerTransformer(method='yeo-johnson')
yj_profit = pt.fit_transform(store['Profit'].values.reshape(-1, 1)).ravel()
print(f'\nSkew Profit raw={stats.skew(store["Profit"]):.2f}, '
      f'Yeo-Johnson={stats.skew(yj_profit):.2f}')

# Part 4: CI on raw vs log scale from a sample of 300
samp = sales.sample(n=300, random_state=5)
lo1, hi1 = stats.t.interval(0.95, df=len(samp)-1, loc=samp.mean(), scale=stats.sem(samp))
log_samp = np.log(samp)
lo2, hi2 = stats.t.interval(0.95, df=len(log_samp)-1, loc=log_samp.mean(), scale=stats.sem(log_samp))
print(f'\n95% CI mean raw Sales : [{lo1:,.1f}, {hi1:,.1f}]')
print(f'95% CI mean log(Sales): [{lo2:.3f}, {hi2:.3f}]')
print('The log-scale CI is more trustworthy: raw Sales is highly skewed, so for a small '
      'sample the mean and SE are dominated by a few extreme orders and the normality '
      'assumption behind the t-interval is shaky. The log scale is closer to Normal.')

<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| Population vs sample | We estimate unknown population parameters from a measurable sample |
| `series.sample(n=...)` | Draw a random sample |
| `stats.sem(data)` | Standard error of the mean (`s / sqrt(n)`); shrinks as n grows |
| `stats.t.interval(0.95, df, loc, scale)` | Confidence interval for a mean |
| `stats.norm / lognorm / poisson` | Normal, Log-Normal, Poisson distributions |
| `.fit()`, `.pdf()`, `.cdf()`, `.ppf()`, `.rvs()` | Estimate params, density, cumulative prob, quantile, random draws |
| `stats.probplot(data, plot=ax)` | Q-Q plot to check a distribution fit |
| `stats.skew(data)` | Measure skewness (0 = symmetric) |
| `np.log1p`, `stats.boxcox`, `PowerTransformer` | Reduce skew (log, Box-Cox, Yeo-Johnson) |
| Transform then back-transform | Apply symmetric-distribution rules (z-scores, thresholds, CIs) correctly to skewed data |


## Conclusion

You can now reason from a sample to a population, quantify the uncertainty with standard errors and confidence intervals, identify the distribution behind a variable, and transform skewed data so standard methods apply. These are general statistical foundations; the next notebook puts them to work in formal hypothesis testing.

<a id="reading"></a>
## Further Reading & Resources

- [SciPy stats reference](https://docs.scipy.org/doc/scipy/reference/stats.html) the full distribution and test catalogue.
- [scikit-learn PowerTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html) Box-Cox and Yeo-Johnson.
- [Penn State STAT 500: Confidence Intervals](https://online.stat.psu.edu/stat500/) a clear, worked refresher.